# MTG Premodern — Scraping de Cartas

Scrapea las listas de cartas de cada mazo desde `deck.php`.

**Prerequisito:** haber corrido `01_scrape_metadata.ipynb` y subir `premodern.db`.

**Método:** `requests` puro con cookie `verificado=1` + `PHPSESSID`. Sin Playwright.

**Campos que se guardan por carta:**
- `card_name`, `quantity`, `is_sideboard` → tabla `deck_cards`
- `card_type`, `cardmarket_id`, `cardmarket_url` → tabla `cards`

**Main vs Sideboard:** misma carta en ambos = dos registros distintos (PK es `deck_id + card_name + is_sideboard`).

**Tiempo estimado:** ~5-6 hs para ~37,000 mazos (delay 0.5 s/request). Resumible: si se corta, volvé a ejecutar la celda 8.

In [ ]:
# Celda 1: Instalar dependencias
!pip install -q requests beautifulsoup4 lxml tqdm

In [ ]:
# Celda 2: Subir la base de datos
import os

DB_FILE = "premodern.db"

if not os.path.exists(DB_FILE):
    try:
        from google.colab import files
        print("Subí el archivo premodern.db:")
        uploaded = files.upload()
    except ImportError:
        print(f"ERROR: No se encontró {DB_FILE}")
else:
    size_mb = os.path.getsize(DB_FILE) / 1024 / 1024
    print(f"{DB_FILE} encontrado ({size_mb:.1f} MB)")

In [ ]:
# Celda 3: Imports y configuración
import sqlite3
import time
import re
import logging
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

BASE_URL  = "https://www.tcdecks.net"
DECK_URL  = f"{BASE_URL}/deck.php"
DELAY     = 0.5   # segundos entre requests
BATCH_SIZE = 200  # commit cada N mazos procesados
MAX_CONSECUTIVE_BLOCKS = 10  # si hay N bloqueos seguidos, reinicia sesión

conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA foreign_keys=ON")

# Asegurarse de que las columnas nuevas existen (por si la DB fue creada antes)
for col, typedef in [("card_type", "TEXT"), ("cardmarket_id", "INTEGER"), ("cardmarket_url", "TEXT")]:
    try:
        conn.execute(f"ALTER TABLE cards ADD COLUMN {col} {typedef}")
        conn.commit()
        print(f"Columna '{col}' agregada a cards.")
    except Exception:
        pass  # Ya existe

total  = conn.execute("SELECT COUNT(*) FROM decks").fetchone()[0]
pend   = conn.execute("SELECT COUNT(*) FROM decks WHERE cards_scraped = 0").fetchone()[0]
done   = total - pend

print(f"Mazos totales:    {total:,}")
print(f"Ya scrapeados:    {done:,}")
print(f"Pendientes:       {pend:,}")
print(f"Tiempo estimado:  ~{pend * DELAY / 3600:.1f} horas")

In [ ]:
# Celda 4: Crear vistas si no existen
#
# v_meta_share        → participación de cada arquetipo por mes
# v_archetype_success → tasa de top8/top4/win por arquetipo
# v_card_popularity   → popularidad de cada carta (main y side por separado)
# v_player_stats      → estadísticas agregadas por jugador

views = {
    "v_meta_share": """
        SELECT
            d.archetype,
            strftime('%Y-%m', d.date) AS month,
            COUNT(*) AS deck_count,
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (
                PARTITION BY strftime('%Y-%m', d.date)
            ) AS meta_share_pct
        FROM decks d
        GROUP BY d.archetype, strftime('%Y-%m', d.date)
    """,

    "v_archetype_success": """
        SELECT
            d.archetype,
            COUNT(*) AS total_entries,
            AVG(CASE WHEN d.position <= 8  THEN 1.0 ELSE 0.0 END) AS top8_rate,
            AVG(CASE WHEN d.position <= 4  THEN 1.0 ELSE 0.0 END) AS top4_rate,
            AVG(CASE WHEN d.position  = 1  THEN 1.0 ELSE 0.0 END) AS win_rate,
            AVG(d.position * 1.0 / d.total_players) AS avg_relative_position
        FROM decks d
        WHERE d.total_players >= 8
        GROUP BY d.archetype
        HAVING COUNT(*) >= 10
    """,

    "v_card_popularity": """
        SELECT
            dc.card_name,
            dc.is_sideboard,
            COUNT(DISTINCT dc.deck_id) AS deck_count,
            SUM(dc.quantity)           AS total_copies,
            AVG(dc.quantity)           AS avg_copies_per_deck
        FROM deck_cards dc
        GROUP BY dc.card_name, dc.is_sideboard
    """,

    "v_player_stats": """
        SELECT
            d.player_name,
            COUNT(*)                                              AS total_entradas,
            COUNT(DISTINCT d.tournament_id)                       AS torneos_jugados,
            COUNT(DISTINCT d.archetype)                           AS arquetipos_distintos,
            SUM(CASE WHEN d.position = 1   THEN 1 ELSE 0 END)    AS victorias,
            SUM(CASE WHEN d.position <= 4  THEN 1 ELSE 0 END)    AS top4s,
            SUM(CASE WHEN d.position <= 8  THEN 1 ELSE 0 END)    AS top8s,
            ROUND(AVG(d.position * 1.0 / d.total_players), 4)    AS avg_posicion_relativa,
            ROUND(
                SUM(CASE WHEN d.position <= 8 THEN 1.0 ELSE 0.0 END) / COUNT(*), 4
            )                                                     AS top8_rate,
            MIN(d.date)                                           AS primer_torneo,
            MAX(d.date)                                           AS ultimo_torneo
        FROM decks d
        WHERE d.total_players >= 8
        GROUP BY d.player_name
        HAVING COUNT(*) >= 3
    """,
}

for name, body in views.items():
    conn.execute(f"DROP VIEW IF EXISTS {name}")
    conn.execute(f"CREATE VIEW {name} AS {body}")

conn.commit()

existing = [
    r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name"
    )
]
print(f"Vistas creadas: {existing}")


In [ ]:
# Celda 5: Inicializar sesión HTTP
#
# Requisitos descubiertos via reverse engineering:
#   1. Cookie verificado=1
#   2. PHPSESSID (se obtiene visitando cualquier página)
#   3. Accept-Encoding: gzip, deflate, br  ← CRÍTICO, sin esto el sitio devuelve 403
#   4. Headers Sec-Fetch-* y Referer

def create_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
        ),
        "Accept": (
            "text/html,application/xhtml+xml,application/xml;"
            "q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8"
        ),
        "Accept-Language":  "en-US,en;q=0.9,es;q=0.8",
        "Accept-Encoding":  "gzip, deflate, br",  # ← SIN ESTO EL SITIO BLOQUEA
        "Connection":       "keep-alive",
        "Sec-Fetch-Dest":   "document",
        "Sec-Fetch-Mode":   "navigate",
        "Sec-Fetch-Site":   "same-origin",
        "Sec-Fetch-User":   "?1",
        "Upgrade-Insecure-Requests": "1",
        "Referer": f"{BASE_URL}/results.php?format=Premodern&src=all&page=1",
    })
    retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.cookies.set("verificado", "1", domain="www.tcdecks.net", path="/")
    # Visitar results.php para obtener PHPSESSID
    s.get(f"{BASE_URL}/results.php?format=Premodern&src=all&page=1", timeout=30)
    return s

session = create_session()
print(f"Sesión iniciada. Cookies: {list(session.cookies.keys())}")

In [ ]:
# Celda 6: Parser de cartas
#
# Estructura HTML confirmada de deck.php (tabla.table_deck, fila con valign=top):
#
#   <td valign=top>  → Mainboard columna izquierda
#     <h6>Creatures [4]</h6>
#     4 <a href="https://cardmarket.com/...?idProduct=8331&utm_..." name="Phyrexian Dreadnought">...
#     <h6>Instants [34]</h6>
#     4 <a href="..." name="Counterspell">...
#     ...
#   <td valign=top>  → Mainboard columna derecha (Sorceries, Artifacts, Lands)
#     <h6>Land [17]</h6>
#     17 <a ... name="Island">...
#   <td valign=top>  → Sideboard (sin headers <h6>)
#     3 <a ... name="Essence Flare">...
#
# Misma carta en main y side → dos registros separados gracias a is_sideboard.

TYPE_MAP = {
    "Creatures":     "Creature",
    "Instants":      "Instant",
    "Sorceries":     "Sorcery",
    "Artifacts":     "Artifact",
    "Enchantments":  "Enchantment",
    "Planeswalkers": "Planeswalker",
    "Land":          "Land",
    "Lands":         "Land",
    "Battle":        "Battle",
}

CARD_RE = re.compile(
    r'(\d+)\s*<a\b[^>]*?href="([^"]+)"[^>]*?name="([^"]+)"[^>]*?>'
)


def parse_deck_html(html, deck_id):
    """Parse deck page HTML. Returns list of card dicts or None if blocked."""
    if "Acceso Denegado" in html or len(html) < 500:
        return None

    soup = BeautifulSoup(html, "lxml")
    deck_table = soup.find("table", class_="table_deck")
    if not deck_table:
        return None

    data_cells = deck_table.find_all("td", valign="top")
    if not data_cells:
        return None

    # Last cell = sideboard (when ≥2 cells exist)
    sideboard_idx = len(data_cells) - 1 if len(data_cells) >= 2 else None

    cards = []

    for cell_idx, cell in enumerate(data_cells):
        is_sideboard = (cell_idx == sideboard_idx)
        cell_html = str(cell)

        # Split on <h6> tags to track card type per section
        parts = re.split(r'(<h6>[^<]*</h6>)', cell_html)
        current_type = None  # Sideboard has no <h6> → stays None

        for part in parts:
            h6 = re.match(r'<h6>([^\[<]+?)(?:\s*\[\d+\])?\s*</h6>', part.strip())
            if h6:
                raw = h6.group(1).strip()
                current_type = TYPE_MAP.get(raw, raw)
                continue

            for m in CARD_RE.finditer(part):
                qty  = int(m.group(1))
                href = m.group(2)
                name = m.group(3).strip()

                cm_id_m = re.search(r'idProduct=(\d+)', href)
                cm_id   = int(cm_id_m.group(1)) if cm_id_m else None
                cm_url  = re.split(r'&utm_', href)[0] if href else None

                cards.append({
                    "deck_id":        deck_id,
                    "card_name":      name,
                    "quantity":       qty,
                    "is_sideboard":   is_sideboard,
                    "card_type":      current_type,  # None for sideboard
                    "cardmarket_id":  cm_id,
                    "cardmarket_url": cm_url,
                })

    return cards if cards else None


print("Parser listo.")

In [ ]:
# Celda 7: TEST — validar parser y sesión con 3 mazos conocidos
import time

TEST_DECKS = [
    (459525, 43670, "Stiflenought",   60, 15),
    (459526, 43670, "Enchantress",    60, 15),
    (459527, 43670, "Landstill",      60, 15),
]

all_ok = True
for deck_id, tourn_id, name, exp_main, exp_side in TEST_DECKS:
    r = session.get(f"{DECK_URL}?id={tourn_id}&iddeck={deck_id}", timeout=30)
    cards = parse_deck_html(r.text, deck_id)

    if cards is None:
        print(f"❌ {name}: BLOQUEADO — {r.text[:80]}")
        all_ok = False
        time.sleep(3)
        continue

    main = [c for c in cards if not c["is_sideboard"]]
    side = [c for c in cards if c["is_sideboard"]]
    main_total = sum(c["quantity"] for c in main)
    side_total = sum(c["quantity"] for c in side)

    # Check main/side overlap (same card in both)
    main_names = {c["card_name"] for c in main}
    side_names = {c["card_name"] for c in side}
    overlap = main_names & side_names

    # Check card_type and cardmarket_id present in mainboard
    main_with_type = sum(1 for c in main if c["card_type"])
    main_with_cm   = sum(1 for c in main if c["cardmarket_id"])

    ok = (main_total == exp_main and side_total == exp_side)
    if not ok:
        all_ok = False

    print(f"{'✅' if ok else '⚠️'} {name}:")
    print(f"   Main: {len(main)} tipos, {main_total} cartas | Side: {len(side)} tipos, {side_total} cartas")
    print(f"   Main con card_type: {main_with_type}/{len(main)} | con cardmarket_id: {main_with_cm}/{len(main)}")
    if overlap:
        print(f"   Cartas en main Y side: {overlap} ← deben ser 2 registros distintos ✅")
    # Show sample
    print(f"   Ejemplo main: {main[0]['quantity']}x {main[0]['card_name']} [{main[0]['card_type']}] cm={main[0]['cardmarket_id']}")
    if side:
        print(f"   Ejemplo side: {side[0]['quantity']}x {side[0]['card_name']} [{side[0]['card_type']}]")
    print()
    time.sleep(1)

print("🚀 Todo OK — continuá con la celda siguiente." if all_ok else "⚠️ Revisar errores antes de continuar.")

In [ ]:
# Celda 8: Limpiar datos malos del intento anterior con Playwright (si los hay)

bad = conn.execute(
    "SELECT COUNT(*) FROM deck_cards WHERE card_name LIKE '%Acceso%'"
).fetchone()[0]

if bad > 0:
    print(f"Encontrados {bad} registros malos del scraping anterior. Limpiando...")
    bad_deck_ids = [
        r[0] for r in conn.execute(
            "SELECT DISTINCT deck_id FROM deck_cards WHERE card_name LIKE '%Acceso%'"
        ).fetchall()
    ]
    # Borrar todas las cartas de esos mazos (el resto también puede ser basura)
    conn.execute("DELETE FROM deck_cards WHERE card_name LIKE '%Acceso%'")
    conn.execute("DELETE FROM cards     WHERE name      LIKE '%Acceso%'")
    for did in bad_deck_ids:
        conn.execute("DELETE FROM deck_cards WHERE deck_id = ?", (did,))
        conn.execute("UPDATE decks SET cards_scraped = 0 WHERE id = ?", (did,))
    conn.commit()
    print(f"✅ Limpiados {len(bad_deck_ids)} mazos — se re-scrapearán.")
else:
    print("No hay datos malos. Todo limpio.")

pend = conn.execute("SELECT COUNT(*) FROM decks WHERE cards_scraped = 0").fetchone()[0]
print(f"Mazos pendientes: {pend:,}")

In [ ]:
# Celda 9 (OPCIONAL): DRY RUN — probar con N mazos antes del scraping completo
#
# Configura cuántos mazos querés probar y si guardar los datos en la DB.
# Podés ejecutar esto cuantas veces quieras antes de lanzar el scraping completo.
#
# Cuando estés conforme → ejecutá la Celda 9 (scraping completo).

N_TEST      = 5        # Cuántos mazos testear
COMMIT_DRY  = True     # True = guardar en DB | False = solo mostrar, no guardar
SAMPLE_MODE = "random" # "random" = aleatorio entre pendientes | "first" = primeros

import random

pending_all = conn.execute(
    "SELECT id, tournament_id FROM decks WHERE cards_scraped = 0"
).fetchall()

if not pending_all:
    print("No hay mazos pendientes.")
else:
    sample = (
        random.sample(pending_all, min(N_TEST, len(pending_all)))
        if SAMPLE_MODE == "random"
        else pending_all[:N_TEST]
    )

    print(f"DRY RUN — {len(sample)} mazos  COMMIT={COMMIT_DRY}")
    print("=" * 60)

    ok = 0
    blocked = 0
    errors  = 0

    for deck_id, tourn_id in sample:
        url = f"{DECK_URL}?id={tourn_id}&iddeck={deck_id}"
        try:
            r = session.get(url, timeout=30)
            cards = parse_deck_html(r.text, deck_id)

            if cards is None:
                blocked += 1
                print(f"  BLOQUEADO  deck_id={deck_id}  HTTP={r.status_code}")
                print(f"             Preview: {r.text[:120]!r}")
                time.sleep(3)
                continue

            main = [c for c in cards if not c["is_sideboard"]]
            side = [c for c in cards if c["is_sideboard"]]
            main_total   = sum(c["quantity"] for c in main)
            side_total   = sum(c["quantity"] for c in side)
            n_types      = len({c["card_type"] for c in main if c["card_type"]})
            missing_type = sum(1 for c in main if not c["card_type"])
            missing_cm   = sum(1 for c in main if not c["cardmarket_id"])

            flag_main = "✅" if 55 <= main_total <= 65 else "⚠️"
            flag_side = "✅" if 0 <= side_total <= 15 else "⚠️"

            print(f"  deck_id={deck_id}  {flag_main} main={main_total}  {flag_side} side={side_total}")
            print(f"             tipos de carta={n_types}  sin tipo={missing_type}  sin CM id={missing_cm}")

            # Muestra hasta 3 cartas de cada zona
            for label, zona in [("main", main[:3]), ("side", side[:3])]:
                for c in zona:
                    print(
                        f"    {label}  {c['quantity']}x {c['card_name']!r:30s}"
                        f"  type={c['card_type']!r}  cm={c['cardmarket_id']}"
                    )
            print()

            if COMMIT_DRY:
                for c in cards:
                    conn.execute(
                        """INSERT INTO cards (name, card_type, cardmarket_id, cardmarket_url)
                           VALUES (?, ?, ?, ?)
                           ON CONFLICT(name) DO UPDATE SET
                             card_type      = COALESCE(cards.card_type,      excluded.card_type),
                             cardmarket_id  = COALESCE(cards.cardmarket_id,  excluded.cardmarket_id),
                             cardmarket_url = COALESCE(cards.cardmarket_url, excluded.cardmarket_url)""",
                        (c["card_name"], c["card_type"], c["cardmarket_id"], c["cardmarket_url"]),
                    )
                    conn.execute(
                        """INSERT OR REPLACE INTO deck_cards
                           (deck_id, card_name, quantity, is_sideboard)
                           VALUES (?, ?, ?, ?)""",
                        (c["deck_id"], c["card_name"], c["quantity"], int(c["is_sideboard"])),
                    )
                conn.execute("UPDATE decks SET cards_scraped = 1 WHERE id = ?", (deck_id,))
                conn.commit()

            ok += 1

        except Exception as e:
            errors += 1
            print(f"  ERROR deck_id={deck_id}: {e}")

        time.sleep(DELAY)

    print("=" * 60)
    saved_str = "y guardados en DB" if COMMIT_DRY else "(no guardados — solo visualización)"
    print(f"Resultado: {ok} OK {saved_str} | {blocked} bloqueados | {errors} errores")
    if not COMMIT_DRY:
        print("💡 Cambia COMMIT_DRY=True para persistir los resultados.")
    print("\nCuando estés conforme → ejecutá la Celda 9 para el scraping completo.")


In [ ]:
# Celda 10: SCRAPING COMPLETO
#
# Resumible: si se interrumpe, re-ejecutar esta celda retoma desde donde quedó.
#
# Upsert en tabla 'cards':
#   - Si la carta ya existe, solo completa campos vacíos (no sobreescribe).
#   - Así card_type del mainboard se preserva aunque luego la carta aparezca
#     en un sideboard (donde no hay headers de tipo).
#
# Tabla 'deck_cards':
#   - PK = (deck_id, card_name, is_sideboard)
#   - La misma carta en main y side genera DOS filas distintas.

pending = conn.execute(
    "SELECT id, tournament_id FROM decks WHERE cards_scraped = 0"
).fetchall()

print(f"Mazos pendientes: {len(pending):,}")
print(f"Tiempo estimado:  ~{len(pending) * DELAY / 3600:.1f} horas")

if not pending:
    print("No hay mazos pendientes.")
else:
    success = 0
    blocked = 0
    errors  = 0
    consecutive_blocks = 0

    pbar = tqdm(pending, desc="Cartas", unit="mazo")

    for deck_id, tourn_id in pbar:
        try:
            r = session.get(
                f"{DECK_URL}?id={tourn_id}&iddeck={deck_id}", timeout=30
            )
            cards = parse_deck_html(r.text, deck_id)

            if cards is None:
                blocked += 1
                consecutive_blocks += 1
                if consecutive_blocks >= MAX_CONSECUTIVE_BLOCKS:
                    print(f"\n⚠️ {MAX_CONSECUTIVE_BLOCKS} bloqueos seguidos — reiniciando sesión...")
                    time.sleep(30)
                    session = create_session()
                    consecutive_blocks = 0
                time.sleep(DELAY)
                continue

            consecutive_blocks = 0

            for c in cards:
                # Upsert card metadata (preserva valores existentes si el nuevo es NULL)
                conn.execute(
                    """INSERT INTO cards (name, card_type, cardmarket_id, cardmarket_url)
                       VALUES (?, ?, ?, ?)
                       ON CONFLICT(name) DO UPDATE SET
                         card_type      = COALESCE(cards.card_type,      excluded.card_type),
                         cardmarket_id  = COALESCE(cards.cardmarket_id,  excluded.cardmarket_id),
                         cardmarket_url = COALESCE(cards.cardmarket_url, excluded.cardmarket_url)""",
                    (c["card_name"], c["card_type"], c["cardmarket_id"], c["cardmarket_url"]),
                )
                # Insertar relación mazo-carta
                # Misma carta en main y side → dos registros distintos
                conn.execute(
                    """INSERT OR REPLACE INTO deck_cards
                       (deck_id, card_name, quantity, is_sideboard)
                       VALUES (?, ?, ?, ?)""",
                    (c["deck_id"], c["card_name"], c["quantity"], int(c["is_sideboard"])),
                )

            conn.execute("UPDATE decks SET cards_scraped = 1 WHERE id = ?", (deck_id,))
            success += 1

            if success % BATCH_SIZE == 0:
                conn.commit()

        except Exception as e:
            errors += 1
            if errors % 100 == 0:
                logger.warning(f"Error #{errors} deck {deck_id}: {e}")

        pbar.set_postfix({"ok": success, "block": blocked, "err": errors})
        time.sleep(DELAY)

    conn.commit()
    print(f"\n{'='*50}")
    print(f"Scraping completado!")
    print(f"  Éxitos:    {success:,}")
    print(f"  Bloqueados: {blocked:,}")
    print(f"  Errores:   {errors:,}")

In [ ]:
# Celda 11: Validación
import pandas as pd

print("=" * 55)
print("VALIDACIÓN")
print("=" * 55)

n_cards    = conn.execute("SELECT COUNT(*) FROM cards").fetchone()[0]
n_dc       = conn.execute("SELECT COUNT(*) FROM deck_cards").fetchone()[0]
n_scraped  = conn.execute("SELECT COUNT(*) FROM decks WHERE cards_scraped = 1").fetchone()[0]
n_pending  = conn.execute("SELECT COUNT(*) FROM decks WHERE cards_scraped = 0").fetchone()[0]

print(f"Cartas únicas:       {n_cards:,}")
print(f"Registros deck_cards:{n_dc:,}")
print(f"Mazos con cartas:    {n_scraped:,}")
print(f"Mazos sin cartas:    {n_pending:,}")

# Datos malos
bad = conn.execute("SELECT COUNT(*) FROM deck_cards WHERE card_name LIKE '%Acceso%'").fetchone()[0]
print(f"\nRegistros malos:     {bad} {'✅' if bad == 0 else '❌ Ejecutar celda 7'}")

# card_type coverage
typed = conn.execute("SELECT COUNT(*) FROM cards WHERE card_type IS NOT NULL").fetchone()[0]
print(f"Cartas con tipo:     {typed:,} / {n_cards:,} ({typed/n_cards*100:.1f}% si n_cards>0)")

# cardmarket_id coverage
cm = conn.execute("SELECT COUNT(*) FROM cards WHERE cardmarket_id IS NOT NULL").fetchone()[0]
print(f"Cartas con CM id:    {cm:,} / {n_cards:,}")

# Same card in main + side
overlap = conn.execute(
    """SELECT COUNT(*) FROM (
         SELECT deck_id, card_name FROM deck_cards
         GROUP BY deck_id, card_name HAVING COUNT(DISTINCT is_sideboard) = 2
       )"""
).fetchone()[0]
print(f"\nCarta en main Y side (misma carta, 2 filas): {overlap:,} casos ✅")

# Promedio mainboard
avg = conn.execute(
    """SELECT AVG(t) FROM (
         SELECT SUM(quantity) AS t FROM deck_cards
         WHERE is_sideboard = 0 GROUP BY deck_id
       )"""
).fetchone()[0]
if avg:
    flag = "✅" if 55 <= avg <= 65 else "⚠️"
    print(f"Promedio main/mazo:  {avg:.1f} cartas {flag}")

print("\nTop 15 cartas más jugadas (mainboard):")
df = pd.read_sql_query(
    """SELECT c.name, c.card_type, c.cardmarket_id,
              COUNT(DISTINCT dc.deck_id) AS decks,
              ROUND(AVG(dc.quantity), 2) AS avg_qty
       FROM deck_cards dc
       JOIN cards c ON c.name = dc.card_name
       WHERE dc.is_sideboard = 0
       GROUP BY c.name
       ORDER BY decks DESC LIMIT 15""",
    conn
)
print(df.to_string(index=False))
print("\nTop 10 jugadores por top8_rate (mín. 10 entradas):")
df_players = pd.read_sql_query(
    """SELECT player_name, total_entradas, torneos_jugados,
              victorias, top4s, top8s,
              ROUND(top8_rate * 100, 1) AS top8_pct,
              arquetipos_distintos,
              primer_torneo, ultimo_torneo
       FROM v_player_stats
       WHERE total_entradas >= 10
       ORDER BY top8_rate DESC, total_entradas DESC
       LIMIT 10""",
    conn
)
print(df_players.to_string(index=False))


In [ ]:
# Celda 11b: Funciones para sincronizar datos de Scryfall
# Funciones para sincronizar datos de Scryfall (metadata completa + precios)
import json as _json_scryfall
import requests as _requests_scryfall

SCRYFALL_COLLECTION_URL = "https://api.scryfall.com/cards/collection"
SCRYFALL_BATCH_SIZE = 75
SCRYFALL_DELAY = 0.1

SCRYFALL_NEW_COLUMNS = [
    ("mana_cost", "TEXT"), ("cmc", "REAL"), ("type_line", "TEXT"),
    ("oracle_text", "TEXT"), ("flavor_text", "TEXT"), ("power", "TEXT"),
    ("toughness", "TEXT"), ("loyalty", "TEXT"), ("colors", "TEXT"),
    ("color_identity", "TEXT"), ("keywords", "TEXT"), ("produced_mana", "TEXT"),
    ("rarity", "TEXT"), ("set_code", "TEXT"), ("set_name", "TEXT"),
    ("released_at", "TEXT"), ("collector_number", "TEXT"), ("layout", "TEXT"),
    ("image_uri", "TEXT"), ("scryfall_id", "TEXT"), ("price_usd", "REAL"),
    ("price_updated_at", "TIMESTAMP"), ("scryfall_raw", "TEXT"),
    ("scryfall_synced_at", "TIMESTAMP"),
]

def ensure_scryfall_schema(conn):
    existing = {row[1] for row in conn.execute("PRAGMA table_info(cards)").fetchall()}
    for col, typedef in SCRYFALL_NEW_COLUMNS:
        if col not in existing:
            conn.execute(f"ALTER TABLE cards ADD COLUMN {col} {typedef}")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS card_price_history (
            card_name TEXT NOT NULL REFERENCES cards(name),
            price_usd REAL,
            fetched_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (card_name, fetched_at)
        )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_price_history_card ON card_price_history(card_name)")
    conn.commit()

def cheapest_usd_price(prices):
    """Precio mas economico entre las versiones devueltas por Scryfall (normal/foil/etched), solo USD."""
    candidates = [prices.get("usd"), prices.get("usd_foil"), prices.get("usd_etched")]
    nums = [float(p) for p in candidates if p is not None]
    return min(nums) if nums else None

def _join_list(values):
    return ",".join(values) if values else ""

def fetch_scryfall_batch(card_names):
    """Consulta Scryfall en lotes de 75 nombres. Devuelve (dict nombre->card_dict, not_found)."""
    results = {}
    not_found = []
    for i in range(0, len(card_names), SCRYFALL_BATCH_SIZE):
        batch = card_names[i:i + SCRYFALL_BATCH_SIZE]
        identifiers = [{"name": n} for n in batch]
        resp = _requests_scryfall.post(
            SCRYFALL_COLLECTION_URL, json={"identifiers": identifiers}, timeout=30
        )
        resp.raise_for_status()
        data = resp.json()
        for card in data.get("data", []):
            results[card.get("name")] = card
        for nf in data.get("not_found", []):
            not_found.append(nf.get("name"))
        time.sleep(SCRYFALL_DELAY)
    return results, not_found

def sync_scryfall_metadata(conn):
    """Trae metadata completa (tipo, texto, colores, precio, imagen, etc.) para cartas nunca sincronizadas."""
    ensure_scryfall_schema(conn)
    pending = [r[0] for r in conn.execute("SELECT name FROM cards WHERE scryfall_id IS NULL").fetchall()]
    print(f"[Scryfall metadata] Cartas pendientes: {len(pending):,}")
    if not pending:
        print("[Scryfall metadata] Nada que actualizar.")
        return

    results, not_found = fetch_scryfall_batch(pending)

    for name, card in results.items():
        mana_cost = card.get("mana_cost", "") or ""
        if not mana_cost and "card_faces" in card:
            faces = card["card_faces"]
            mana_cost = "//".join(f.get("mana_cost", "") or "" for f in faces)
        price = cheapest_usd_price(card.get("prices", {}) or {})

        conn.execute(
            """UPDATE cards SET
                 mana_cost = ?, cmc = ?, type_line = ?, oracle_text = ?, flavor_text = ?,
                 power = ?, toughness = ?, loyalty = ?, colors = ?, color_identity = ?,
                 keywords = ?, produced_mana = ?, rarity = ?, set_code = ?, set_name = ?,
                 released_at = ?, collector_number = ?, layout = ?, image_uri = ?,
                 scryfall_id = ?, price_usd = ?, price_updated_at = CURRENT_TIMESTAMP,
                 scryfall_raw = ?, scryfall_synced_at = CURRENT_TIMESTAMP
               WHERE name = ?""",
            (mana_cost, card.get("cmc"), card.get("type_line"), card.get("oracle_text"),
             card.get("flavor_text"), card.get("power"), card.get("toughness"), card.get("loyalty"),
             _join_list(card.get("colors")), _join_list(card.get("color_identity")),
             _join_list(card.get("keywords")), _join_list(card.get("produced_mana")),
             card.get("rarity"), card.get("set"), card.get("set_name"), card.get("released_at"),
             card.get("collector_number"), card.get("layout"),
             (card.get("image_uris") or {}).get("normal"), card.get("id"), price,
             _json_scryfall.dumps(card), name),
        )
        if price is not None:
            conn.execute(
                "INSERT OR REPLACE INTO card_price_history (card_name, price_usd, fetched_at) "
                "VALUES (?, ?, CURRENT_TIMESTAMP)",
                (name, price),
            )
    conn.commit()
    print(f"[Scryfall metadata] Actualizadas: {len(results):,}")
    print(f"[Scryfall metadata] No encontradas: {len(not_found)}")
    if not_found:
        print("[Scryfall metadata] Ejemplos no encontrados:", not_found[:20])

def refresh_scryfall_prices(conn):
    """Refresca el precio mas economico de TODAS las cartas ya sincronizadas y
    guarda una fila nueva en card_price_history (para trackear evolucion en el tiempo)."""
    ensure_scryfall_schema(conn)
    synced = [r[0] for r in conn.execute("SELECT name FROM cards WHERE scryfall_id IS NOT NULL").fetchall()]
    print(f"[Scryfall precios] Cartas a refrescar: {len(synced):,}")
    if not synced:
        print("[Scryfall precios] Nada que refrescar (corre sync_scryfall_metadata primero).")
        return

    results, not_found = fetch_scryfall_batch(synced)
    updated = 0
    for name, card in results.items():
        price = cheapest_usd_price(card.get("prices", {}) or {})
        if price is None:
            continue
        conn.execute(
            "UPDATE cards SET price_usd = ?, price_updated_at = CURRENT_TIMESTAMP WHERE name = ?",
            (price, name),
        )
        conn.execute(
            "INSERT OR REPLACE INTO card_price_history (card_name, price_usd, fetched_at) "
            "VALUES (?, ?, CURRENT_TIMESTAMP)",
            (name, price),
        )
        updated += 1
    conn.commit()
    print(f"[Scryfall precios] Precios actualizados: {updated:,}")
    if not_found:
        print(f"[Scryfall precios] No encontradas: {len(not_found)}")


In [ ]:
# Celda 11c: Sincronizar metadata + precios de las cartas scrapeadas
# Sincronizar metadata de cartas nuevas + refrescar precios de todas (idempotente)
sync_scryfall_metadata(conn)
refresh_scryfall_prices(conn)


In [ ]:
# Celda 12: Descargar base de datos actualizada

# Checkpoint WAL + verificación de integridad antes de descargar
print("Aplicando WAL checkpoint...")
conn.execute("PRAGMA wal_checkpoint(TRUNCATE)")
conn.commit()

ic = conn.execute("PRAGMA integrity_check").fetchone()[0]
if ic == "ok":
    print("✅ integrity_check: ok — el archivo está listo para descargar")
else:
    print(f"⚠️  integrity_check: {ic[:200]}")
    print("   Revisá el scraping antes de descargar")
conn.close()

try:
    from google.colab import files
    files.download(DB_FILE)
    print(f"Descargando {DB_FILE}...")
except ImportError:
    print(f"No estás en Colab. El archivo está en: {DB_FILE}")